## Импорт нужных модулей и библиотек

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from collections import Counter
from sklearn.neighbors import NearestNeighbors
import json
from langdetect import detect, DetectorFactory

## Загрузка датасета

In [2]:
DATA_PATH = "../data/raw/coursera_courses.csv"
df = pd.read_csv(DATA_PATH)

## Препроцессинг

### Оставляем только курсы на английском языке

Ибо в датасете есть курсы, переведенные на другой язык, что засоряет рекомендации.

In [3]:
DetectorFactory.seed = 0

def detect_language(text):
    """
    Detects language of the text.
    """
    if pd.isna(text) or text == "":
        return "unknown"
    try:
        return detect(text)
    except:
        return "unknown"

# Detect language to each course.
df["language"] = df["course_title"].apply(detect_language)

print("Distribution of languages:")
print(df["language"].value_counts())

Distribution of languages:
language
en    819
nl     25
es     24
it     21
fr     21
ca     19
de     15
da     12
tl     11
af      6
cy      4
no      4
id      4
ja      3
ro      3
ar      2
pt      2
lt      1
ko      1
et      1
so      1
uk      1
Name: count, dtype: int64


In [4]:
# Deleting non-english languages.
df = df[df["language"] == "en"].copy()
df = df.drop(columns=["language"])
df = df.reset_index(drop=True)

### Очистка данных от ненужных символов

In [5]:
df["students_clean"] = df["course_students_enrolled"].str.replace(",", "").str.replace("k", "000").astype(float)
df["reviews_clean"] = df["course_reviews_num"].str.replace(",", "").str.replace("k", "000").astype(float)

### Парсинг ячеек

In [6]:
def parse_skills(skill_str):
    """
    Turns string to array of skills.
    """
    if pd.isna(skill_str) or skill_str == "":
        return []
    try:
        if skill_str.startswith("[") and skill_str.endswith("]"):
            return ast.literal_eval(skill_str)
    except:
        pass

    try:
        cleaned = skill_str.strip("[]").strip()
        skills = [s.strip().strip("'").strip('"') for s in cleaned.split(",") if s.strip()]
        return skills
    except:
        pass

    return []

### Кодирование категорий числами

In [7]:
difficulty_map = {
    "Beginner": 0,
    "Intermediate": 1,
    "Mixed": 2,
    "Advanced": 3
}

cert_map = {
    "Course": 0,
    "Specialization": 1,
    "Professional Certificate": 2,
    "Guided Project": 3
}

duration_map = {
    "Less Than 2 Hours": 0,
    "1 - 4 Weeks": 1,
    "1 - 3 Months": 2,
    "3 - 6 Months": 3
}

### Применение изменений к датасету

In [8]:
preprocessed_df = df.copy()

In [9]:
# Applying changes to useful columns to make features.
preprocessed_df["skills_list"] = preprocessed_df["course_skills"].apply(parse_skills)
preprocessed_df["num_skills"] = preprocessed_df["course_skills"].apply(len)
preprocessed_df["difficulty_encoded"] = preprocessed_df["course_difficulty"].map(difficulty_map)
preprocessed_df["cert_encoded"] = preprocessed_df["course_certificate_type"].map(cert_map)
preprocessed_df["duration"] = preprocessed_df["course_time"].map(duration_map)

## Формирование признакового пространства

Превращаем курсы в числа, чтобы k‑NN мог искать похожие.

- Берём 100 самых частых навыков. Каждый курс → вектор из 0 и 1 (есть навык / нет).
**Зачем:** основа похожести курсов.

- Формируем TF‑IDF описаний
Текст описаний → 100 чисел через TF‑IDF.
**Зачем:** ловим смысл, который не попал в навыки.

## Загрузка категорий для признаков из файла .json

Формирую категории, чтобы объединить тысячи разных названий навыков (Python Programming, Python Scripting, Basic Python Syntax) в одну общую группу "Python". Так модель видит не 500+ отдельных навыков, а 30-40 осмысленных блоков. Это решает проблему редких тем (SQL, Medical, Biology) — они перестают теряться в шуме и попадают в рекомендации.

Каждый курс получает счетчик по каждой категории: сколько навыков из этой группы у него есть. Например, если у курса 3 навыка из Data Science, он получит 3, а не 1. Так модель точнее понимает, насколько курс связан с темой.

Категории предварительно были собраны с помощью LLM модели и сохранены в categories.json для удобства и чистоты кода.

In [10]:
with open("../data/preprocessed/categories.json", "r", encoding="utf-8") as f:
    categories_json = json.load(f)

# Creating dict to turn skills to categories.
skill_to_category = {}
for category, skills in categories_json.items():
    for skill in skills:
        skill_to_category[skill.lower()] = category

# Catrgory namings
category_columns = list(categories_json.keys())
def map_skill_to_category(skill):
    """Turns skill into a category"""
    skill_lower = skill.lower()
    return skill_to_category.get(skill_lower, skill)


# Creating matrix of skill counts.
category_matrix_counts = pd.DataFrame(0, index=preprocessed_df.index, columns=category_columns)

for idx, skills in enumerate(preprocessed_df["skills_list"]):
    for skill in skills:
        category = map_skill_to_category(skill)
        if category in category_columns:
            category_matrix_counts.loc[idx, category] += 1

for col in category_columns:
    preprocessed_df[col] = category_matrix_counts[col]

In [11]:
# Deleting useless columns.
columns_to_drop = [
    "course_title",
    "course_organization",
    "course_url",
    "course_reviews_num",
    "course_students_enrolled",
    "course_summary",
    "course_description",
    "course_certificate_type",
    "course_time",
    "course_difficulty",
    "course_skills",
    "skills_list"
]
preprocessed_df = preprocessed_df.drop(columns_to_drop, axis=1)

In [12]:
preprocessed_df.head()

,course_rating,students_clean,reviews_clean,num_skills,difficulty_encoded,cert_encoded,duration,Data Science & Analytics,Statistics,Data Visualization,...,Finance & Accounting,Web Development,UX/UI Design,Healthcare & Medicine,Engineering,Music & Audio,Game Development,Career Development & Job Preparation,Research Methods & Case Studies,English Language
0,4.7,6958.0,492.0,151,0,1,3,0,0,0,...,1,0,0,0,0,0,0,0,0,0
1,4.3,2531.0,51.0,190,1,1,2,0,0,0,...,0,10,0,0,0,0,0,0,0,0
2,4.8,4377.0,62.0,2,0,0,2,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4.7,39004.0,517.0,87,1,0,2,0,3,0,...,0,0,0,0,0,0,0,0,0,0
4,NaN,NaN,NaN,115,0,0,2,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Устранение пропусков, NAN значений

In [13]:
if "course_rating" in preprocessed_df.columns:
    preprocessed_df["course_rating"] = preprocessed_df["course_rating"].fillna(4.5)

if "students_clean" in preprocessed_df.columns:
    preprocessed_df["students_clean"] = preprocessed_df["students_clean"].fillna(0)

if "reviews_clean" in preprocessed_df.columns:
    preprocessed_df["reviews_clean"] = preprocessed_df["reviews_clean"].fillna(0)

## Формирование признакового пространства

In [14]:
X = preprocessed_df.values

numpy.ndarray

## Формирование топ-15 популярных курсов для рекомендаций

In [15]:
TOP_POPULAR = 15
popular_courses = df.nlargest(TOP_POPULAR, "students_clean").index.tolist()

## Уравновешиваем признаки

Некоторые признаки имеют слишком большой вес - рейтинг, например, не должен играть слишком большую роль в рекомендации.

In [16]:
X_weighted = X.copy()

weights = {
    "rating": 0.5,
    "num_skills": 0.0,
    "difficulty": 2.5,
    "cert": 0.5,
    "duration": 0.5,
    "categories": 7.0,
}

num_base = 5
num_categories = len(category_columns)

base_cols = ["rating", "num_skills", "difficulty", "cert", "duration"]
for i, col in enumerate(base_cols):
    X_weighted[:, i] *= weights[col]

X_weighted[:, num_base:num_base + num_categories] *= weights["categories"]

## StandardScaler для масштабирования пространства признаков

In [17]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_weighted)

In [18]:
print(X_scaled)

[[ 0.13351891  0.         -0.06561466 ... -0.16315175 -0.10035907
  -0.11625885]
 [-1.96940385  0.         -0.15998151 ... -0.16315175 -0.10035907
  -0.11625885]
 [ 0.65924959  0.         -0.15762769 ... -0.16315175 -0.10035907
  -0.11625885]
 ...
 [ 1.18498028  0.         -0.16916141 ... -0.16315175 -0.10035907
   3.16705137]
 [ 1.18498028  0.          0.00328811 ... -0.16315175 -0.10035907
  -0.11625885]
 [-0.39221178  0.         -0.11761272 ... -0.16315175 -0.10035907
  -0.11625885]]


## Класс Recommender, отвечающий за подбор рекомендаций

Модель работает так: каждый курс в датасете превращается в числовой вектор — около 40 признаков. Основу составляют категории навыков.

Когда пользователь передает список пройденных курсов, модель берет векторы этих курсов и усредняет их — так получается единый портрет пользователя.

Затем алгоритм ищет в датасете 12 ближайших курсов к этому портрету, используя косинусное расстояние. Из найденных соседей исключаются уже пройденные курсы, и берутся 7 лучших. К ним добавляется один случайный курс из топ-15 популярных, чтобы внести разнообразие. Итого пользователь получает 8 рекомендаций.

Если пользователь новый и ничего не проходил, модель просто отдает 8 случайных популярных курсов — это называется холодным стартом.

In [19]:
import numpy as np
import pandas as pd
from collections import Counter

class Recommender:
    """
    K-NN based course recommender system.
    Returns 8 recommendations: 7 from k-NN + 1 random from top-15 popular courses.
    For new users (cold start) returns 8 random popular courses.
    """

    def __init__(self, data, df, knn, popular_courses):
        self.data = data
        self.df = df
        self.knn = knn
        self.popular_courses = popular_courses

    def get_user_vector(self, course_ids):
        """
        Builds user profile vector by averaging embeddings of their completed courses.
        Returns zero vector if no courses provided.
        """
        if not course_ids:
            return np.zeros(self.data.shape[1])
        return np.mean(self.data[course_ids], axis=0)

    def find_similar_users(self, user_vector, k_neighbors=12):
        """
        Finds k-nearest neighbors in the latent space using cosine distance.
        Returns distances and indices of neighbors.
        """
        distances, indices = self.knn.kneighbors(
            user_vector.reshape(1, -1),
            n_neighbors=k_neighbors
        )
        return distances[0], indices[0]

    def get_recommendations(self, user_course_ids, top_n=8):
        """
        Main recommendation logic.
        - Cold start: returns random popular courses
        - Existing user: 7 k-NN + 1 random popular (excludes user's courses)
        """
        if not user_course_ids:
            sampled = np.random.choice(
                self.popular_courses,
                size=min(top_n, len(self.popular_courses)),
                replace=False
            )
            results = self.df.iloc[sampled][
                ["course_title", "course_organization", "course_difficulty", "course_rating"]
            ].copy()
            results["source"] = "Popular (cold start)"
            results["score"] = 0
            return results

        user_vector = self.get_user_vector(user_course_ids)
        distances, indices = self.find_similar_users(user_vector, k_neighbors=12)

        neighbor_courses = []
        for idx in indices:
            neighbor_courses.append(idx)

        course_counter = Counter(neighbor_courses)

        for course_id in user_course_ids:
            if course_id in course_counter:
                del course_counter[course_id]

        knn_course_ids = [
            course_id for course_id, _ in course_counter.most_common(12)
            if course_id not in user_course_ids
        ][:7]

        used_indices = set(user_course_ids)
        used_indices.update(knn_course_ids)

        available_popular = [c for c in self.popular_courses if c not in used_indices]
        popular_selected = []
        if available_popular:
            popular_selected.append(np.random.choice(available_popular))

        all_course_ids = knn_course_ids + popular_selected

        if len(all_course_ids) < top_n:
            for course_id in self.popular_courses:
                if course_id not in used_indices:
                    all_course_ids.append(course_id)
                    used_indices.add(course_id)
                    if len(all_course_ids) >= top_n:
                        break

        results = self.df.iloc[all_course_ids][
            ["course_title", "course_organization", "course_difficulty", "course_rating"]
        ].copy()

        sources = []
        for i in range(len(all_course_ids)):
            if i < 7:
                sources.append("k-NN")
            else:
                sources.append("Popular (random)")

        results["source"] = sources
        results["score"] = [course_counter.get(course_id, 0) for course_id in all_course_ids]

        return results

    def recommend_by_titles(self, course_titles):
        """
        Convenience method: accepts course titles instead of indices.
        Returns recommendations for the given titles.
        """
        if not course_titles:
            return self.get_recommendations([])

        course_ids = []
        for title in course_titles:
            matches = self.df[self.df["course_title"].str.contains(title, case=False)]
            if len(matches) > 0:
                course_ids.append(matches.index[0])

        if not course_ids:
            return self.get_recommendations([])

        return self.get_recommendations(course_ids)

## Создание KNN модели, опирающейся на 6 соседей.

In [20]:
knn_model = NearestNeighbors(n_neighbors=10, metric="euclidean", algorithm="brute")
knn_model.fit(X_scaled)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",10
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'euclidean'
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'eu...an'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


## Создание экземпляра модели рекомендаций

In [21]:
user_rec = Recommender(X_scaled, df, knn_model, popular_courses)

## Функция для удобного отображения рекомендаций

In [22]:
def show_recommendations(course_titles):
    """
    Shows recommendations for user.
    """
    print(f"User completed courses:")
    for title in course_titles:
        print(f"  - {title}")

    recs = user_rec.recommend_by_titles(course_titles)

    print("\nRecomendations:")
    for i, (idx, row) in enumerate(recs.iterrows(), 1):
        print(f"{i}. {row['course_title'][:50]}...")

## Тесты-примеры системы рекомендаций

In [23]:
show_recommendations([
    "Crash Course on Python",
    "Python Programming Fundamentals",
    "Python for Data Science, AI & Development",
    "Python 3 Programming",
    "Python Basics"
])

User completed courses:
  - Crash Course on Python
  - Python Programming Fundamentals
  - Python for Data Science, AI & Development
  - Python 3 Programming
  - Python Basics

Recomendations:
1. Programming for Everybody (Getting Started with Py...
2. Automate Cybersecurity Tasks with Python...
3. Using Python to Interact with the Operating System...
4. Python Data Structures...
5. Python Functions, Files, and Dictionaries...
6. Biology Meets Programming: Bioinformatics for Begi...
7. Using Python to Access Web Data...
8. Calculus for Machine Learning and Data Science...


In [24]:
show_recommendations([])

User completed courses:

Recomendations:
1. Python Data Structures...
2. Learning How to Learn: Powerful mental tools to he...
3. Calculus for Machine Learning and Data Science...
4. Speak English Professionally: In Person, Online & ...
5. Foundations of Public Health Practice...
6. Foundations: Data, Data, Everywhere...
7. What is Data Science?...
8. Creative Thinking: Techniques and Tools for Succes...


In [1]:
show_recommendations([
    "Data Science Methodology",
    "Data Science Math Skills",
    "Data Science: Foundations using R"
])

NameError: name 'show_recommendations' is not defined

In [26]:
show_recommendations([
    "Foundations of Project Management",
    "Project Initiation: Starting a Successful Project",
    "Project Execution: Running the Project",
    "Project Launch"
])

User completed courses:
  - Foundations of Project Management
  - Project Initiation: Starting a Successful Project
  - Project Execution: Running the Project
  - Project Launch

Recomendations:
1. Capstone: Applying Project Management in the Real ...
2. Project Management: Foundations and Initiation...
3. Fundamentals of Project Planning and Management...
4. Project Planning: Putting It All Together...
5. Construction Project Management...
6. Agile with Atlassian Jira...
7. Project Management Foundations, Initiation, and Pl...
8. Creative Thinking: Techniques and Tools for Succes...


In [46]:
show_recommendations([".NET FullStack Developer"])

User completed courses:
  - .NET FullStack Developer

Recomendations:
1. Java Programming and Software Engineering Fundamen...
2. Introductory C Programming...
3. Software Testing and Automation...
4. Java FullStack Developer...
5. Object Oriented Programming in Java...
6. Introduction to Git and GitHub...
7. Web Design for Everybody: Basics of Web Developmen...
8. Python Data Structures...


## Метрики

#### Бейзлайны для сравнения

In [27]:
def baseline_popular(user_course_ids, df, top_n=8):
    """Baseline: recommends the most popular courses."""
    popular = df.nlargest(top_n * 3, "students_clean")
    popular = popular[~popular.index.isin(user_course_ids)]
    return popular.head(top_n).index.tolist()

def baseline_random(user_course_ids, df, top_n=8):
    """Baseline: recommends random courses."""
    available = df[~df.index.isin(user_course_ids)]
    return available.sample(min(top_n, len(available))).index.tolist()

In [28]:
def get_relevant_courses(user_course_ids, df, skill_matrix, threshold=1):
    """
    Searchs for intersections between courses.
    """
    if len(user_course_ids) == 0:
        return []

    user_categories = skill_matrix.loc[user_course_ids].sum(axis=0)

    relevant = []
    for idx in df.index:
        if idx in user_course_ids:
            continue
        course_categories = skill_matrix.loc[idx]
        overlap = np.minimum(user_categories, course_categories).sum()
        if overlap >= threshold:
            relevant.append(idx)

    return relevant

#### Метрика map@k

**Что показывает:** Насколько хорошо релевантные курсы расположены в списке.

**Зачем:** Учитывает не только факт попадания, но и позицию — чем выше, тем лучше.

In [29]:
def map_at_k(recommended, relevant, k=8):
    """
    Mean Average Precision@k
    """
    if not relevant:
        return 0.0

    recommended_k = recommended[:k]
    score = 0.0
    num_hits = 0.0

    for i, item in enumerate(recommended_k):
        if item in relevant:
            num_hits += 1.0
            score += num_hits / (i + 1.0)

    return score / min(len(relevant), k)

#### Precision@k

**Что показывает:** Сколько из 8 рекомендованных курсов реально подходят пользователю.

**Зачем**: Оценивает точность модели — не рекомендует ли она лишнее.

In [30]:
def precision_at_k(recommended, relevant, k=8):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / k

#### Recall@k

**Что показывает:** Какую долю от всех возможных релевантных курсов мы нашли.

**Зачем:** Оценивает полноту — не упускаем ли мы хорошие варианты.

In [40]:
def recall_at_k(recommended, relevant, k=8):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / len(relevant) if len(relevant) > 0 else 0

#### Hit@k

**Что показывает:** Есть ли вообще хоть один релевантный курс среди 8 рекомендаций.

**Зачем:** Проверяет, не провалилась ли модель полностью.

In [31]:
def hit_rate_at_k(recommended, relevant, k=8):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return 1.0 if hits > 0 else 0.0

## Тесты

#### Тест-кейсы

In [37]:
test_scenarios = [
    {
        "name": "Python (5 курсов)",
        "courses": [
            "Crash Course on Python",
            "Python Programming Fundamentals",
            "Python for Data Science, AI & Development",
            "Python 3 Programming",
            "Python Basics"
        ]
    },
    {
        "name": "Data Science (3 курса)",
        "courses": [
            "Data Science Methodology",
            "Data Science Math Skills",
            "Data Science: Foundations using R"
        ]
    },
    {
        "name": "Project Management (4 курса)",
        "courses": [
            "Foundations of Project Management",
            "Project Initiation: Starting a Successful Project",
            "Project Execution: Running the Project",
            "Project Launch"
        ]
    }
]

In [41]:
all_results = []

for scenario in test_scenarios:
    print(f"\nCase: {scenario['name']}")

    if scenario["courses"]:
        user_course_ids = user_rec.recommend_by_titles(scenario["courses"]).index.tolist()
    else:
        user_course_ids = []

    knn_recs = user_rec.get_recommendations(user_course_ids)
    knn_ids = knn_recs.index.tolist()

    popular_ids = baseline_popular(user_course_ids, df, 8)
    random_ids = baseline_random(user_course_ids, df, 8)

    if user_course_ids:
        relevant = get_relevant_courses(user_course_ids, df, category_matrix_counts, threshold=1)
    else:
        relevant = []

    print(f"Relevant: {len(relevant)}")

    methods = {
        "k-NN": knn_ids,
        "Popular": popular_ids,
        "Random": random_ids
    }

    results = {}
    for name, recs in methods.items():
        results[name] = {
            "MAP@8": map_at_k(recs, relevant, 8),
            "Precision@8": precision_at_k(recs, relevant, 8),
            "Recall@8": recall_at_k(recs, relevant, 8),
            "Hit Rate@8": hit_rate_at_k(recs, relevant, 8),
        }

    print("\nMetrics:")
    print("-"*70)
    print(f"{'Method':<12} {'MAP@8':<10} {'Prec@8':<10} {'Recall@8':<10} {'Hit@8':<10}")
    print("-"*70)
    for name, metrics in results.items():
        print(f"{name:<12} {metrics['MAP@8']:.4f}    {metrics['Precision@8']:.4f}    {metrics['Recall@8']:.4f}    {metrics['Hit Rate@8']:.4f}    ")

    all_results.append({
        "scenario": scenario["name"],
        **{f"{name}_{metric}": value for name, metrics in results.items() for metric, value in metrics.items()}
    })

print("\nAverage for all cases")

df_results = pd.DataFrame(all_results)

print(f"\n{'Metrics':<20} {'k-NN':<12} {'Popular':<12} {'Random':<12}")
print("-"*60)

avg_metrics = {
    "MAP@8": ("MAP@8", "MAP@8", "MAP@8"),
    "Precision@8": ("Precision@8", "Precision@8", "Precision@8"),
    "Recall@8": ("Recall@8", "Recall@8", "Recall@8"),
    "Hit Rate@8": ("Hit Rate@8", "Hit Rate@8", "Hit Rate@8"),
}

for metric, (knn_col, popular_col, random_col) in avg_metrics.items():
    knn_avg = df_results[f"k-NN_{knn_col}"].mean()
    popular_avg = df_results[f"Popular_{popular_col}"].mean()
    random_avg = df_results[f"Random_{random_col}"].mean()
    print(f"{metric:<20} {knn_avg:.4f}     {popular_avg:.4f}     {random_avg:.4f}")


Case: Python (5 курсов)
Relevant: 202

Metrics:
----------------------------------------------------------------------
Method       MAP@8      Prec@8     Recall@8   Hit@8     
----------------------------------------------------------------------
k-NN         0.6250    0.6250    0.0248    1.0000    
Popular      0.0250    0.1250    0.0050    1.0000    
Random       0.2083    0.2500    0.0099    1.0000    

Case: Data Science (3 курса)
Relevant: 252

Metrics:
----------------------------------------------------------------------
Method       MAP@8      Prec@8     Recall@8   Hit@8     
----------------------------------------------------------------------
k-NN         0.7500    0.7500    0.0238    1.0000    
Popular      0.1994    0.5000    0.0159    1.0000    
Random       0.0625    0.1250    0.0040    1.0000    

Case: Project Management (4 курса)
Relevant: 250

Metrics:
----------------------------------------------------------------------
Method       MAP@8      Prec@8     Recall@8 

Модель показала высокую точность: **Precision@8 = 79.2%** — это значит, что в среднем **6-7 из 8** рекомендованных курсов действительно релевантны пользователю. Но все метрики меняются в зависимости от запуска, ведь используется один рандомный курс.

**MAP@8 = 0.792** подтверждает, что релевантные курсы находятся на правильных позициях (первые 1-2 места). При этом наша модель превосходит Popular-бейзлайн в **3.4 раза** по MAP (0.792 против 0.232) и в **1.9 раза** по Precision (0.792 против 0.417), что доказывает эффективность персонализации.

**Recall@8** ожидаемо низкий (~2.3%), так как мы рекомендуем всего 8 курсов из сотен релевантных, доступных в датасете — это нормально для топ-N рекомендаций.

**Hit Rate@8 = 100%** — во всех тестовых случаях модель нашла хотя бы один релевантный курс в топ-8, что говорит о её стабильности.

In [45]:
import joblib

model_artifacts = {
    "data": X_scaled,
    "knn_model": knn_model,
    "popular_courses": popular_courses,
    "scaler": scaler,
    "category_columns": category_columns,
    "X_weighted": X_weighted if 'X_weighted' in locals() else None
}

joblib.dump(model_artifacts, "../models/knn_recommender_final.pkl")
print("Model is saeved '../models/knn_recommender_final.pkl'")

Model is saeved '../models/knn_recommender_final.pkl'
